In [0]:
spark.sql("SELECT 'Databricks is ready!'")


DataFrame['Databricks is ready!': string]

In [0]:
# List files in the volume folder to confirm upload
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/foodbank_data"))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/foodbank_data/visits_generated.csv,visits_generated.csv,708036,1765086969000


In [0]:
from pyspark.sql import functions as F

src_csv = "dbfs:/Volumes/workspace/default/foodbank_data/visits_generated.csv"  

visits_raw = spark.read.option("header", True).option("inferSchema", True).csv(src_csv)
visits_raw = visits_raw.withColumn("ingestion_time", F.current_timestamp())

display(visits_raw.limit(10))
print("Columns:", visits_raw.columns)
print("Row count (quick sample count may be slow):", visits_raw.count())


visit_id,client_id,visit_date,location_id,household_size,items_received_count,category,visit_type,source_channel,ingestion_time
V1,C229,2024-11-01,BocaRaton,1,3,Dry,walk-in,referral,2025-12-07T06:01:10.421Z
V2,C1517,2024-11-01,BocaRaton,4,2,Hygiene,walk-in,referral,2025-12-07T06:01:10.421Z
V3,C477,2024-11-01,BocaRaton,2,1,Dry,walk-in,self,2025-12-07T06:01:10.421Z
V4,C860,2024-11-01,BocaRaton,1,4,Canned,walk-in,referral,2025-12-07T06:01:10.421Z
V5,C866,2024-11-01,BocaRaton,2,2,Fresh,walk-in,referral,2025-12-07T06:01:10.421Z
V6,C736,2024-11-01,BocaRaton,4,4,Canned,walk-in,self,2025-12-07T06:01:10.421Z
V7,C1993,2024-11-01,BocaRaton,4,2,Fresh,appointment,self,2025-12-07T06:01:10.421Z
V8,C1765,2024-11-01,BocaRaton,2,2,Canned,walk-in,referral,2025-12-07T06:01:10.421Z
V9,C593,2024-11-01,BocaRaton,5,5,Canned,walk-in,referral,2025-12-07T06:01:10.421Z
V10,C1709,2024-11-01,BocaRaton,2,3,Dry,walk-in,self,2025-12-07T06:01:10.421Z


Columns: ['visit_id', 'client_id', 'visit_date', 'location_id', 'household_size', 'items_received_count', 'category', 'visit_type', 'source_channel', 'ingestion_time']
Row count (quick sample count may be slow): 12116


In [0]:
# Ensure visit_date is proper date, and types are OK
visits_clean = (visits_raw
                .withColumn("visit_date", F.to_date("visit_date", "yyyy-MM-dd"))
                .withColumn("household_size", F.coalesce(F.col("household_size").cast("int"), F.lit(1)))
                .withColumn("items_received_count", F.coalesce(F.col("items_received_count").cast("int"), F.lit(1)))
               )
 
display(visits_clean.limit(5))


visit_id,client_id,visit_date,location_id,household_size,items_received_count,category,visit_type,source_channel,ingestion_time
V1,C229,2024-11-01,BocaRaton,1,3,Dry,walk-in,referral,2025-12-07T06:02:04.045Z
V2,C1517,2024-11-01,BocaRaton,4,2,Hygiene,walk-in,referral,2025-12-07T06:02:04.045Z
V3,C477,2024-11-01,BocaRaton,2,1,Dry,walk-in,self,2025-12-07T06:02:04.045Z
V4,C860,2024-11-01,BocaRaton,1,4,Canned,walk-in,referral,2025-12-07T06:02:04.045Z
V5,C866,2024-11-01,BocaRaton,2,2,Fresh,walk-in,referral,2025-12-07T06:02:04.045Z


In [0]:
bronze_path = "dbfs:/Volumes/workspace/default/foodbank_data/bronze/visits"

visits_clean.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(bronze_path)
print("Bronze Delta written to:", bronze_path)


Bronze Delta written to: dbfs:/Volumes/workspace/default/foodbank_data/bronze/visits


In [0]:
# path you wrote to earlier
bronze_path = "dbfs:/Volumes/workspace/default/foodbank_data/bronze/visits"

# read the delta files directly
df = spark.read.format("delta").load(bronze_path)
display(df.limit(5))
print("rows:", df.count())

# create a temporary SQL view you can query in this session
df.createOrReplaceTempView("bronze_visits_temp")
# now you can run SQL against bronze_visits_temp
display(spark.sql("SELECT location_id, count(*) as cnt FROM bronze_visits_temp GROUP BY location_id ORDER BY cnt DESC"))


visit_id,client_id,visit_date,location_id,household_size,items_received_count,category,visit_type,source_channel,ingestion_time
V1,C229,2024-11-01,BocaRaton,1,3,Dry,walk-in,referral,2025-12-07T06:02:44.067Z
V2,C1517,2024-11-01,BocaRaton,4,2,Hygiene,walk-in,referral,2025-12-07T06:02:44.067Z
V3,C477,2024-11-01,BocaRaton,2,1,Dry,walk-in,self,2025-12-07T06:02:44.067Z
V4,C860,2024-11-01,BocaRaton,1,4,Canned,walk-in,referral,2025-12-07T06:02:44.067Z
V5,C866,2024-11-01,BocaRaton,2,2,Fresh,walk-in,referral,2025-12-07T06:02:44.067Z


rows: 12116


location_id,cnt
BocaRaton,3517
Delray,3499
Deerfield,2830
Boynton,2270


In [0]:
# =========================
# SILVER LAYER 
# =========================


In [0]:
bronze_path = "dbfs:/Volumes/workspace/default/foodbank_data/bronze/visits"

df_bronze = spark.read.format("delta").load(bronze_path)
display(df_bronze.limit(5))


visit_id,client_id,visit_date,location_id,household_size,items_received_count,category,visit_type,source_channel,ingestion_time
V1,C229,2024-11-01,BocaRaton,1,3,Dry,walk-in,referral,2025-12-07T06:02:44.067Z
V2,C1517,2024-11-01,BocaRaton,4,2,Hygiene,walk-in,referral,2025-12-07T06:02:44.067Z
V3,C477,2024-11-01,BocaRaton,2,1,Dry,walk-in,self,2025-12-07T06:02:44.067Z
V4,C860,2024-11-01,BocaRaton,1,4,Canned,walk-in,referral,2025-12-07T06:02:44.067Z
V5,C866,2024-11-01,BocaRaton,2,2,Fresh,walk-in,referral,2025-12-07T06:02:44.067Z


In [0]:
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    .withColumn("visit_date", F.to_date("visit_date"))
    .withColumn("household_size", F.col("household_size").cast("int"))
    .withColumn("items_received_count", F.col("items_received_count").cast("int"))
    .withColumn("location_id", F.trim(F.col("location_id")))
)
display(df_silver.limit(5))


visit_id,client_id,visit_date,location_id,household_size,items_received_count,category,visit_type,source_channel,ingestion_time
V1,C229,2024-11-01,BocaRaton,1,3,Dry,walk-in,referral,2025-12-07T06:02:44.067Z
V2,C1517,2024-11-01,BocaRaton,4,2,Hygiene,walk-in,referral,2025-12-07T06:02:44.067Z
V3,C477,2024-11-01,BocaRaton,2,1,Dry,walk-in,self,2025-12-07T06:02:44.067Z
V4,C860,2024-11-01,BocaRaton,1,4,Canned,walk-in,referral,2025-12-07T06:02:44.067Z
V5,C866,2024-11-01,BocaRaton,2,2,Fresh,walk-in,referral,2025-12-07T06:02:44.067Z


In [0]:
silver_path = "dbfs:/Volumes/workspace/default/foodbank_data/silver/visits"

df_silver.write.format("delta").mode("overwrite").save(silver_path)

print("Silver layer created at:", silver_path)


Silver layer created at: dbfs:/Volumes/workspace/default/foodbank_data/silver/visits


In [0]:
df_check = spark.read.format("delta").load(silver_path)
display(df_check.limit(5))
print("Rows in Silver:", df_check.count())


visit_id,client_id,visit_date,location_id,household_size,items_received_count,category,visit_type,source_channel,ingestion_time
V1,C229,2024-11-01,BocaRaton,1,3,Dry,walk-in,referral,2025-12-07T06:02:44.067Z
V2,C1517,2024-11-01,BocaRaton,4,2,Hygiene,walk-in,referral,2025-12-07T06:02:44.067Z
V3,C477,2024-11-01,BocaRaton,2,1,Dry,walk-in,self,2025-12-07T06:02:44.067Z
V4,C860,2024-11-01,BocaRaton,1,4,Canned,walk-in,referral,2025-12-07T06:02:44.067Z
V5,C866,2024-11-01,BocaRaton,2,2,Fresh,walk-in,referral,2025-12-07T06:02:44.067Z


Rows in Silver: 12116


In [0]:
silver_df = spark.read.format("delta").load(
    "dbfs:/Volumes/workspace/default/foodbank_data/silver/visits"
)

display(silver_df)


visit_id,client_id,visit_date,location_id,household_size,items_received_count,category,visit_type,source_channel,ingestion_time
V1,C229,2024-11-01,BocaRaton,1,3,Dry,walk-in,referral,2025-12-07T06:02:44.067Z
V2,C1517,2024-11-01,BocaRaton,4,2,Hygiene,walk-in,referral,2025-12-07T06:02:44.067Z
V3,C477,2024-11-01,BocaRaton,2,1,Dry,walk-in,self,2025-12-07T06:02:44.067Z
V4,C860,2024-11-01,BocaRaton,1,4,Canned,walk-in,referral,2025-12-07T06:02:44.067Z
V5,C866,2024-11-01,BocaRaton,2,2,Fresh,walk-in,referral,2025-12-07T06:02:44.067Z
V6,C736,2024-11-01,BocaRaton,4,4,Canned,walk-in,self,2025-12-07T06:02:44.067Z
V7,C1993,2024-11-01,BocaRaton,4,2,Fresh,appointment,self,2025-12-07T06:02:44.067Z
V8,C1765,2024-11-01,BocaRaton,2,2,Canned,walk-in,referral,2025-12-07T06:02:44.067Z
V9,C593,2024-11-01,BocaRaton,5,5,Canned,walk-in,referral,2025-12-07T06:02:44.067Z
V10,C1709,2024-11-01,BocaRaton,2,3,Dry,walk-in,self,2025-12-07T06:02:44.067Z


In [0]:
from pyspark.sql.functions import year, weekofyear

weekly_df = (
    silver_df
    .withColumn("year", year("visit_date"))
    .withColumn("week", weekofyear("visit_date"))
)


In [0]:
from pyspark.sql.functions import avg

avg_household_df = (
    weekly_df
    .groupBy("year", "week", "location_id")
    .agg(avg("household_size").alias("avg_household_size"))
    .orderBy("year", "week", "location_id")
)

display(avg_household_df)


year,week,location_id,avg_household_size
2024,1,BocaRaton,2.1774193548387095
2024,1,Boynton,2.425
2024,1,Deerfield,2.3333333333333335
2024,1,Delray,2.4634146341463414
2024,44,BocaRaton,2.393548387096774
2024,44,Boynton,2.5436893203883497
2024,44,Deerfield,2.25
2024,44,Delray,2.538961038961039
2024,45,BocaRaton,2.4072164948453607
2024,45,Boynton,2.407258064516129
